# 🩸 Phase 3c — WBC-Only Triple Fusion (v3.2)
**PneumoFusionNet · Scaleup Dataset (~3,763 PA-view images)**

> **Ablation Study**: Can a single clinical biomarker — White Blood Cell (WBC) count —
> capture most of the diagnostic signal provided by the full 17-feature MIMIC-IV panel?

---

## Hypothesis
WBC is the primary **acute-phase biomarker of bacterial infection**.
As the **top-ranked permutation-importance feature** in Phase 3, a WBC-only model should:
- Maintain AUC close to the full 17-feature Phase 3 model.
- Drastically reduce missingness risk (CBC is **always** collected in ED triage).
- Provide a **clinically interpretable**, deployable screening tool.

---

## Architecture

```
[Chest X-ray]  → DenseNet-121 + CBAM  →  1024-d Image Embedding ──────────┐
                                                                             ├─→ 8-Head Cross-Attention
[Report Text]  → Bio_ClinicalBERT     →  768-d Token Sequence  ────────────┘   (512-d attended)
  (FINDINGS + HISTORY; IMPRESSION removed; diagnostic keywords redacted)             │
                                                            ┌─────────────────────────┘
[WBC Count]    → MLP (1→128→128→64)   →  64-d WBC Embedding ──────────────┐
                                                                            │
                                [img(1024) + attn(512) + wbc(64) = 1600-d] │
                                              ↓                             │
                              LayerNorm → Dropout → MLP → 2 logits ←───────┘
```

---

## Phase Comparison  (Scaleup ~3,763 images)

| Phase | Modality | AUC | Sensitivity | Specificity | Accuracy |
|:------|:---------|:---:|:-----------:|:-----------:|:--------:|
| **Phase 1** — Image only | CXR | 0.8258 | 71.0% | — | 76.3% |
| **Phase 2v2** — Image + Text | CXR + Report | 0.9460 | 90.3% | 89.1% | 87.8% |
| **Phase 3** — Full 17 features | CXR + Report + Labs/Vitals | 0.9890 | 89.1% | 94.3% | 91.7% |
| **Phase 3c** *(this notebook)* | CXR + Report + **WBC only** | **0.9712** | **92.3%** | **94.0%** | **93.1%** |

---

## Scaleup Series Notebooks
- `Phase-1.1v4-crossval_tta_PA_Scaleup.ipynb` → Phase 1 image-only baseline
- `Phase-2v2-multimodal_improved_PA_Upscaled.ipynb` → Phase 2 image+text
- `Phase-3c-wbc_only_fusion_V3.2.ipynb` ← **This notebook**

## Cell 0 — Imports & Reproducibility

In [ ]:
# ── Cell 0: Imports & Reproducibility ──────────────────────────────────────────
import os, random, warnings, json, copy, re
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import AutoTokenizer, AutoModel
import torchxrayvision as xrv

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, f1_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')
print('Imports done.')

## Cell 1 — Configuration & Paths (Scaleup ~3,763 images)
All paths reference the **Scaleup** outputs produced by:
- `Phase-1.1v4-crossval_tta_PA_Scaleup.ipynb`
- `Phase-2v2-multimodal_improved_PA_Upscaled.ipynb`

In [ ]:
# ── Cell 1: Configuration & Paths ───────────────────────────────────────────────
MAIN_DIR    = r'C:\2026\PneumoFusionNet\mimic\main'
DATASET_DIR = os.path.join(MAIN_DIR, 'dataset')

# Input data — single merged Scaleup CSV (~3,763 samples)
PAIRED_CSV = os.path.join(DATASET_DIR, 'phase3_paired_scaleup_final.csv')
BBOX_CSV   = os.path.join(MAIN_DIR, 'outputs', 'Phase_1.1v4_PA_crossval_scaleup', 'lung_bboxes.csv')

# Pretrained weights
P1_CKPT   = os.path.join(MAIN_DIR, 'outputs', 'Phase_1.1v4_PA_crossval_scaleup', 'best_model_fold5.pth')
P2V2_CKPT = os.path.join(MAIN_DIR, 'outputs', 'Phase_2v2_Scaleup', 'best_v2_model.pth')

# Output directory
SAVE_DIR = os.path.join(MAIN_DIR, 'outputs', 'Phase_3c_wbc_only_scaleup')
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Model hyperparameters ──────────────────────────────────────────────────────
IMG_SIZE       = 224
MAX_TEXT_LEN   = 256
CLINBERT_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'
IMG_FEAT_DIM   = 1024   # DenseNet-121 + CBAM output
TXT_FEAT_DIM   = 768    # Bio_ClinicalBERT hidden dim
ATTN_DIM       = 512    # Cross-attention projection
ATTN_HEADS     = 8
META_HIDDEN    = 128
META_OUT_DIM   = 64

FUSED_DIM = IMG_FEAT_DIM + ATTN_DIM + META_OUT_DIM   # 1024+512+64 = 1600

# ── Training hyperparameters ──────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
BATCH_SIZE  = 16
LR_FUSION   = 2e-4
LR_BERT     = 1e-5
LR_META     = 1e-3
EPOCHS      = 40
PATIENCE    = 8
FOCAL_GAMMA = 2.0
MIXUP_ALPHA = 0.2
CLASSES     = ['NORMAL', 'PNEUMONIA']
MEAN = [0.5020]; STD = [0.2703]

# WBC-ONLY ablation (1 clinical feature)
CLINICAL_FEATURES   = ['wbc']
N_CLINICAL_FEATURES = 1

# ── Sanity checks ──────────────────────────────────────────────────────────────
print(f'Paired CSV      : {os.path.exists(PAIRED_CSV)}  ->  {PAIRED_CSV}')
print(f'BBox CSV        : {os.path.exists(BBOX_CSV)}')
print(f'P1 Checkpoint   : {os.path.exists(P1_CKPT)}')
print(f'P2v2 Checkpoint : {os.path.exists(P2V2_CKPT)}')
print(f'Save Dir        : {SAVE_DIR}')
print(f'Fused dim       : {FUSED_DIM}  (img={IMG_FEAT_DIM} + attn={ATTN_DIM} + wbc={META_OUT_DIM})')
print(f'Clinical Feats  : {N_CLINICAL_FEATURES}  {CLINICAL_FEATURES}')

## Cell 2 — Data Loading & Merging
Loads the single merged Scaleup CSV. Resolves relative image/report paths.
Uses embedded `raw_report` column when available to avoid repeated disk I/O.

In [ ]:
# ── Cell 2: Data Loading & Merging ──────────────────────────────────────────────

def resolve_path(p):
    if pd.isna(p) or str(p).strip() == '': return ''
    p = str(p).replace('/', os.sep).replace('\\', os.sep)
    if os.path.isabs(p): return p
    return os.path.join(DATASET_DIR, p)

df = pd.read_csv(PAIRED_CSV)
df['image_path']  = df['image_path'].apply(resolve_path)
df['report_path'] = df['report_path'].apply(resolve_path)

# Use embedded raw_report if available (avoids disk I/O)
if 'raw_report' in df.columns:
    df['report_no_impression'] = df['raw_report'].fillna('').astype(str)
elif 'report_no_impression' in df.columns:
    df['report_no_impression'] = df['report_no_impression'].fillna('').astype(str)
else:
    def _read_report(path):
        try:
            with open(path, encoding='utf-8', errors='ignore') as f: return f.read().strip()
        except: return ''
    df['report_no_impression'] = df['report_path'].apply(_read_report)

print(f'Total rows     : {len(df):,}')
print(f'Images on disk : {df["image_path"].apply(os.path.exists).sum():,}/{len(df):,}')
print(f'Reports on disk: {df["report_path"].apply(os.path.exists).sum():,}/{len(df):,}')
print(f'Label distribution:')
print(df['label'].value_counts().rename({0: "Normal", 1: "Pneumonia"}).to_string())

## Cell 3 — Anti-Leakage Text Extraction

**Protocol** (identical to Phase 2v2 and Phase 3 Full):

1. **Section parsing**: Extract only `FINDINGS` and `HISTORY` sections.  
   The `IMPRESSION` (final radiologist diagnosis) is **completely stripped**.
2. **Keyword redaction**: Explicit diagnostic terms (`pneumonia`, `no acute findings`,
   `consistent with`, etc.) are replaced with `[REDACTED]`.

This forces Bio_ClinicalBERT to learn from radiological **observations** rather than reading
the final written diagnosis — mirroring real ED triage where no final report exists yet.

In [ ]:
# ── Cell 3: Anti-Leakage Text Extraction (FINDINGS + HISTORY) ───────────────────

LEAKAGE_RE = re.compile(
    r'\bpneumonia\b|\bpneumonic\b|\bno[ -]acute[ -]\w+'
    r'|\bno finding\w*|\bcompatible with\b|\bconsistent with\b'
    r'|\bnormal study\b|\bno significant\b',
    re.IGNORECASE
)

def extract_rich_text(text):
    parts = []
    h = re.search(r'HISTORY[:\s]+(.*?)(?=FINDINGS|TECHNIQUE|COMPARISON|\n\n|\Z)',
                  text, re.DOTALL | re.IGNORECASE)
    if h: parts.append(h.group(1).strip())
    f = re.search(r'FINDINGS[:\s]+(.*?)(?=IMPRESSION|CONCLUSION|\n\n|\Z)',
                  text, re.DOTALL | re.IGNORECASE)
    if f: parts.append(f.group(1).strip())
    result = ' '.join(parts).strip()
    if not result: result = text[:512]     # fallback: first 512 chars
    return LEAKAGE_RE.sub('[REDACTED]', result)

def read_report_from_disk(path):
    try:
        with open(path, encoding='utf-8', errors='ignore') as f: return f.read().strip()
    except: return ''

has_report = df['report_path'].apply(os.path.exists)
print(f'Reports available on disk: {has_report.sum():,}/{len(df):,}')

df['full_report'] = df['report_path'].apply(read_report_from_disk)
df['report_rich'] = df['full_report'].apply(extract_rich_text)

avg_words = df['report_rich'].str.split().str.len().mean()
print(f'Average words per cleaned report : {avg_words:.1f}')
print('Text extraction done -- IMPRESSION removed, diagnostic keywords redacted.')

## Cell 4 — WBC Clinical Feature Validation
Locates and validates the WBC column. Handles merged CSV naming variants (`wbc`, `wbc_x`, `wbc_y`).
Missing values will be imputed with the **train-set median** after the split (next cell).

In [ ]:
# ── Cell 4: WBC Clinical Feature Validation ─────────────────────────────────────

if 'wbc' not in df.columns:
    if 'wbc_x' in df.columns:
        df['wbc'] = df['wbc_x'].combine_first(df.get('wbc_y', pd.Series(dtype=float)))
    else:
        wbc_like = [c for c in df.columns if 'wbc' in c.lower()]
        if wbc_like:
            df['wbc'] = df[wbc_like[0]]
        else:
            raise KeyError('No WBC column found in dataset.')

CLINICAL_FEATURES   = ['wbc']
N_CLINICAL_FEATURES = 1

print(f'Rows              : {len(df):,}')
print(f'WBC missing count : {df["wbc"].isna().sum():,}  '
      f'({df["wbc"].isna().mean()*100:.1f}%)')
print(f'WBC range         : [{df["wbc"].min():.1f}, {df["wbc"].max():.1f}]  '
      f'mean={df["wbc"].mean():.2f}  median={df["wbc"].median():.2f}')
print(f'\nLabel distribution:')
print(df['label'].value_counts().rename({0: 'Normal', 1: 'Pneumonia'}).to_string())
print('\nNote: WBC will be imputed with train-set median AFTER split (no leakage).')

## Cell 5 — Stratified Train / Val / Test Split + StandardScaler

Split: **70% Train / 15% Val / 15% Test** (stratified by pneumonia label).  
WBC standardised with `StandardScaler` **fit only on train** to prevent leakage.  
Missing WBC values imputed with **train-set median** before scaling.

In [ ]:
# ── Cell 5: Train / Val / Test Split (70/15/15, stratified) + StandardScaler ────

train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_frac, stratify=train_val_df['label'], random_state=SEED)

train_df = train_df.copy(); val_df = val_df.copy(); test_df = test_df.copy()

# Impute missing WBC with train-set median (fit on train only)
train_wbc_median = float(train_df['wbc'].median())
if pd.isna(train_wbc_median): train_wbc_median = 8.0
for split_df in [train_df, val_df, test_df]:
    split_df['wbc'] = split_df['wbc'].fillna(train_wbc_median).astype(float)

# Standardise WBC (Z-score, fit on train only -- no data leakage)
scaler = StandardScaler()
train_df[CLINICAL_FEATURES] = scaler.fit_transform(train_df[CLINICAL_FEATURES])
val_df[CLINICAL_FEATURES]   = scaler.transform(val_df[CLINICAL_FEATURES])
test_df[CLINICAL_FEATURES]  = scaler.transform(test_df[CLINICAL_FEATURES])

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n0 = (split['label']==0).sum(); n1 = (split['label']==1).sum()
    print(f'{name:5s}: {len(split):5,}  (Normal={n0:,}, Pneumonia={n1:,})')

print(f'\nWBC after scaling -- mean={train_df["wbc"].mean():.4f}, std={train_df["wbc"].std():.4f}')
print('Split + StandardScaler done. No data leakage.')

## Cell 6 — Lung Bounding-Box Lookup & Triple-Modal Dataset

`TripleModalCXRDataset` returns a 5-tuple per sample:
```python
(image_tensor, token_ids, attention_mask, wbc_scalar_tensor, label)
```

Image preprocessing:
- **CLAHE** contrast enhancement on grayscale CXR
- Optional lung **bounding-box crop** (pre-computed by Phase 1)
- Train: RandomHorizontalFlip + RandomRotation(8°) + ColorJitter + Normalize
- Val/Test: Resize + Normalize only

In [ ]:
# ── Cell 6: Bbox Lookup & TripleModalCXRDataset ─────────────────────────────────

bbox_df     = pd.read_csv(BBOX_CSV)
bbox_lookup = bbox_df.set_index('image_path').to_dict('index')
print(f'Bbox entries: {len(bbox_lookup):,}')

class TripleModalCXRDataset(Dataset):
    def __init__(self, df, bbox_lookup, tokenizer, img_transform,
                 clinical_features, max_len=MAX_TEXT_LEN):
        self.df                = df.reset_index(drop=True)
        self.bbox_lookup       = bbox_lookup
        self.tokenizer         = tokenizer
        self.transform         = img_transform
        self.clinical_features = clinical_features
        self.max_len           = max_len
        self.clahe             = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Image
        img = cv2.imread(row.image_path, cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((224, 224), dtype=np.uint8)
        img = self.clahe.apply(img)
        bb  = self.bbox_lookup.get(row.image_path, None)
        if bb and bb.get('x_max', 0) > 0:
            img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
        img = self.transform(Image.fromarray(img))

        # Text
        enc = self.tokenizer(
            row.report_rich, max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )

        # WBC scalar
        meta = torch.tensor(
            [float(row[c]) for c in self.clinical_features], dtype=torch.float32
        )
        return (img, enc['input_ids'].squeeze(0), enc['attention_mask'].squeeze(0),
                meta, int(row.label))

    def set_transform(self, t): self.transform = t


# Image transforms
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
print('Dataset class and transforms ready.')

## Cell 7 — Model Architecture (WBC-Only TripleFusionNet)

| Module | Role | Frozen? |
|--------|------|---------|
| `ImageEncoder` (DenseNet-121 + CBAM) | Extracts 1024-d visual features | ✅ Frozen |
| `TextEncoder` (Bio_ClinicalBERT) | Encodes FINDINGS+HISTORY tokens | Partly — last 2 layers fine-tuned |
| `WBCMetadataEncoder` (MLP 1→128→128→64) | Encodes WBC scalar to 64-d | Trained from scratch |
| `TripleFusionNet` | 8-Head CrossAttn + 1600-d MLP head | CrossAttn warm-started from P2v2 |

In [ ]:
# ── Cell 7: Model Architecture ───────────────────────────────────────────────────

class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1); self.max = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(nn.Conv2d(c, c//r, 1, bias=False), nn.ReLU(True),
                                 nn.Conv2d(c//r, c, 1, bias=False))
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.mlp(self.avg(x)) + self.mlp(self.max(x)))

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k//2, bias=False); self.sig = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.conv(torch.cat(
            [x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], 1)))

class CBAM(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.ca = ChannelAttention(c, r); self.sa = SpatialAttention()
    def forward(self, x): return self.sa(self.ca(x))


class ImageEncoder(nn.Module):
    def __init__(self, ckpt):
        super().__init__()
        xrv_m = xrv.models.DenseNet(weights='densenet121-res224-all')
        self.features = xrv_m.features; self.cbam = CBAM(1024)
        self.pool = nn.AdaptiveAvgPool2d(1)
        sd = {k: v for k, v in torch.load(ckpt, map_location='cpu').items()
              if k.startswith('features.') or k.startswith('cbam.')}
        miss, unexp = self.load_state_dict(sd, strict=False)
        print(f'[ImageEncoder] loaded {len(sd)} keys | missing={len(miss)} unexpected={len(unexp)}')
        for p in self.parameters(): p.requires_grad = False
    def forward(self, x):
        f = F.relu(self.features(x), True)
        return self.pool(self.cbam(f)).flatten(1)   # (B, 1024)


class TextEncoder(nn.Module):
    def __init__(self, model_name=CLINBERT_MODEL):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for name, p in self.bert.named_parameters():
            p.requires_grad = any(
                tag in name for tag in ['encoder.layer.10', 'encoder.layer.11', 'pooler'])
        trainable = sum(p.numel() for p in self.bert.parameters() if p.requires_grad)
        print(f'[TextEncoder] Unfrozen BERT params: {trainable:,}')
    def forward(self, input_ids, attention_mask):
        return self.bert(input_ids=input_ids,
                         attention_mask=attention_mask).last_hidden_state  # (B, seq, 768)


class WBCMetadataEncoder(nn.Module):
    # MLP: 1 -> 128 -> 128 -> 64  (same output dim as Phase 3 Full, so FUSED_DIM stays 1600)
    def __init__(self, in_dim=N_CLINICAL_FEATURES, hidden=META_HIDDEN, out_dim=META_OUT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, out_dim), nn.ReLU()
        )
        print(f'[WBCMetadataEncoder] in_dim={in_dim} -> out_dim={out_dim}  '
              f'params={sum(p.numel() for p in self.net.parameters()):,}')
    def forward(self, x): return self.net(x)   # (B, 64)


class TripleFusionNet(nn.Module):
    # WBC-Only Triple Fusion:
    #   img(1024) + cross_attn(512) + wbc(64) = 1600-d -> MLP -> 2 logits
    #   Cross-attn: image queries BERT token sequence (same as Phase 2v2 + Phase 3)
    #   Classifier head warm-started from Phase 2v2 cross-attn keys
    def __init__(self, img_dim=IMG_FEAT_DIM, txt_dim=TXT_FEAT_DIM,
                 attn_dim=ATTN_DIM, heads=ATTN_HEADS, meta_out=META_OUT_DIM):
        super().__init__()
        self.query_proj = nn.Linear(img_dim, attn_dim)
        self.key_proj   = nn.Linear(txt_dim, attn_dim)
        self.val_proj   = nn.Linear(txt_dim, attn_dim)
        self.cross_attn = nn.MultiheadAttention(attn_dim, heads, batch_first=True, dropout=0.1)
        self.norm1      = nn.LayerNorm(attn_dim)
        fused_dim = img_dim + attn_dim + meta_out    # 1600
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim), nn.Dropout(0.4),
            nn.Linear(fused_dim, 512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 128),       nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
        print(f'[TripleFusionNet] fused_dim={fused_dim}  (img={img_dim} + attn={attn_dim} + wbc={meta_out})')
    def forward(self, img_feat, txt_tokens, meta_feat):
        q = self.query_proj(img_feat).unsqueeze(1)        # (B,1,512)
        k = self.key_proj(txt_tokens)                      # (B,seq,512)
        v = self.val_proj(txt_tokens)                      # (B,seq,512)
        attended, _ = self.cross_attn(q, k, v)             # (B,1,512)
        attended = self.norm1(attended.squeeze(1))          # (B,512)
        fused = torch.cat([img_feat, attended, meta_feat], dim=1)  # (B,1600)
        return self.classifier(fused)

print('All model classes defined.')

## Cell 8 — Focal Loss & Triple Mixup Utilities

**Focal Loss** (γ=2.0): Down-weights easy negatives; focuses training on hard boundary cases.  
**Triple Mixup**: Convex interpolation applied *simultaneously* across all 3 embedding branches
(image, text, WBC), preventing any single modality from dominating.

In [ ]:
# ── Cell 8: Focal Loss & Triple Mixup ────────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, weight=None):
        super().__init__()
        self.gamma = gamma; self.weight = weight
    def forward(self, logits, labels):
        ce   = F.cross_entropy(logits, labels, weight=self.weight, reduction='none')
        pt   = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def mixup_triple(img_feat, txt_tokens, meta_feat, labels, alpha=MIXUP_ALPHA):
    if alpha <= 0:
        return img_feat, txt_tokens, meta_feat, labels, labels, 1.0
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(img_feat.size(0), device=img_feat.device)
    return (
        lam * img_feat   + (1 - lam) * img_feat[idx],
        lam * txt_tokens + (1 - lam) * txt_tokens[idx],
        lam * meta_feat  + (1 - lam) * meta_feat[idx],
        labels, labels[idx], lam
    )

def mixup_criterion(criterion, logits, la, lb, lam):
    return lam * criterion(logits, la) + (1 - lam) * criterion(logits, lb)

print('Focal Loss and Triple Mixup ready.')

## Cell 9 — Instantiate Models, Load Checkpoints, Build DataLoaders

Weight loading strategy:
- **ImageEncoder**: Phase 1 Fold-5 Scaleup checkpoint → **frozen**
- **TextEncoder**: Phase 2v2 ClinicalBERT weights → last 2 layers fine-tuned
- **TripleFusionNet**: Cross-attn keys warm-started from Phase 2v2; classifier re-initialised (1536→1600-d)
- **WBCMetadataEncoder**: Random init (trained from scratch)

In [ ]:
# ── Cell 9: Instantiate Models, Load Checkpoints, Build DataLoaders ─────────────

print('Loading Bio_ClinicalBERT tokenizer ...')
tokenizer    = AutoTokenizer.from_pretrained(CLINBERT_MODEL)
text_encoder = TextEncoder(CLINBERT_MODEL).to(DEVICE)

print('\nLoading Phase-1 ImageEncoder (Fold 5 Scaleup) ...')
image_encoder = ImageEncoder(P1_CKPT).to(DEVICE)

print(f'\nBuilding WBCMetadataEncoder (in_dim={N_CLINICAL_FEATURES}) ...')
meta_encoder = WBCMetadataEncoder(in_dim=N_CLINICAL_FEATURES,
                                   hidden=META_HIDDEN, out_dim=META_OUT_DIM).to(DEVICE)

print('\nBuilding TripleFusionNet (WBC-Only) ...')
fusion_model = TripleFusionNet(meta_out=META_OUT_DIM).to(DEVICE)

# Load Phase 2v2 cross-attention weights
print(f'\nLoading Phase 2v2 checkpoint: {os.path.exists(P2V2_CKPT)}')
if os.path.exists(P2V2_CKPT):
    p2v2_state = torch.load(P2V2_CKPT, map_location='cpu')
    if 'text' in p2v2_state:
        miss, unexp = text_encoder.load_state_dict(p2v2_state['text'], strict=False)
        print(f'  [TextEncoder]  missing={len(miss)}  unexpected={len(unexp)}')
    if 'fusion' in p2v2_state:
        ca_keys = {k: v for k, v in p2v2_state['fusion'].items()
                   if any(k.startswith(pfx) for pfx in
                          ['query_proj', 'key_proj', 'val_proj', 'cross_attn', 'norm1'])}
        fusion_model.load_state_dict(ca_keys, strict=False)
        print(f'  [FusionNet]    Loaded {len(ca_keys)} cross-attn keys from P2v2')
        print(f'  Classifier head re-init: 1536-d (P2v2) -> 1600-d (Phase 3c)')
else:
    print('  WARNING: P2v2 checkpoint not found -- training from scratch.')

# Param summary
print(f'\nTrainable parameters:')
print(f'  TripleFusionNet     : {sum(p.numel() for p in fusion_model.parameters() if p.requires_grad):>12,}')
print(f'  TextEncoder (BERT)  : {sum(p.numel() for p in text_encoder.parameters() if p.requires_grad):>12,}')
print(f'  WBCMetadataEncoder  : {sum(p.numel() for p in meta_encoder.parameters() if p.requires_grad):>12,}')
print(f'  ImageEncoder        : {sum(p.numel() for p in image_encoder.parameters() if p.requires_grad):>12,}  (frozen)')

# DataLoaders
train_ds = TripleModalCXRDataset(train_df, bbox_lookup, tokenizer, train_tfm, CLINICAL_FEATURES)
val_ds   = TripleModalCXRDataset(val_df,   bbox_lookup, tokenizer, val_tfm,   CLINICAL_FEATURES)
test_ds  = TripleModalCXRDataset(test_df,  bbox_lookup, tokenizer, val_tfm,   CLINICAL_FEATURES)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'\nDataLoaders ready | Train={len(train_loader)} | Val={len(val_loader)} | Test={len(test_loader)} batches')

## Cell 10 — Training Loop

| Optimiser | Learning Rate |
|-----------|--------------|
| ClinicalBERT (last 2 layers) | `LR_BERT = 1e-5` |
| Fusion cross-attn + classifier | `LR_FUSION = 2e-4` |
| WBC MLP encoder | `LR_META = 1e-3` |

Scheduler: `CosineAnnealingLR`  
Early stopping: patience=8 on validation AUC  
Best model saved to `SAVE_DIR/best_p3c_model.pth`

In [ ]:
# ── Cell 10: Training Loop ────────────────────────────────────────────────────────

def eval_epoch(fusion, img_enc, txt_enc, meta_enc, loader, criterion, device):
    fusion.eval(); img_enc.eval(); txt_enc.eval(); meta_enc.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, ids, masks, meta, labels in loader:
            imgs, ids, masks = imgs.to(device), ids.to(device), masks.to(device)
            meta, labels     = meta.to(device), labels.to(device)
            img_f  = img_enc(imgs)
            txt_t  = txt_enc(ids, masks)
            meta_f = meta_enc(meta)
            logits = fusion(img_f, txt_t, meta_f)
            loss   = criterion(logits, labels)
            probs  = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            total_loss += loss.item() * labels.size(0)
            all_probs.extend(probs); all_labels.extend(labels.cpu().tolist())
    preds = [1 if p >= 0.5 else 0 for p in all_probs]
    acc   = accuracy_score(all_labels, preds)
    auc   = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.5
    return total_loss / len(all_labels), acc, auc, all_probs, all_labels

def find_optimal_threshold(labels, probs):
    fpr, tpr, thresh = roc_curve(labels, probs)
    return float(thresh[np.argmax(tpr - fpr)])

def find_clinical_threshold(labels, probs, target_sens=0.90):
    fpr, tpr, thresh = roc_curve(labels, probs)
    for t, s in zip(thresh, tpr):
        if s >= target_sens: return float(t)
    return find_optimal_threshold(labels, probs)

# Class-weighted Focal Loss
n0 = (train_df['label']==0).sum(); n1 = (train_df['label']==1).sum()
weight = torch.tensor([n1/(n0+n1), n0/(n0+n1)], dtype=torch.float).to(DEVICE)
criterion_focal = FocalLoss(gamma=FOCAL_GAMMA, weight=weight)

# AdamW with 3 LR groups
optimizer = torch.optim.AdamW([
    {'params': text_encoder.parameters(),  'lr': LR_BERT,   'weight_decay': 1e-4},
    {'params': fusion_model.parameters(),  'lr': LR_FUSION, 'weight_decay': 1e-4},
    {'params': meta_encoder.parameters(),  'lr': LR_META,   'weight_decay': 1e-3},
])
scheduler    = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_auc     = 0.0; best_state = None; patience_cnt = 0
history      = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': []}

print('=' * 58)
print('  Phase 3c (WBC-Only) -- Training')
print('=' * 58)

for epoch in range(1, EPOCHS + 1):
    fusion_model.train(); image_encoder.eval()
    text_encoder.train(); meta_encoder.train()
    ep_loss, ep_probs, ep_labels = 0.0, [], []

    for imgs, ids, masks, meta, labels in train_loader:
        imgs, ids, masks = imgs.to(DEVICE), ids.to(DEVICE), masks.to(DEVICE)
        meta, labels     = meta.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            img_f = image_encoder(imgs)
        txt_t  = text_encoder(ids, masks)
        meta_f = meta_encoder(meta)
        img_m, txt_m, meta_m, la, lb, lam = mixup_triple(img_f, txt_t, meta_f, labels)
        optimizer.zero_grad()
        logits = fusion_model(img_m, txt_m, meta_m)
        loss   = mixup_criterion(criterion_focal, logits, la, lb, lam)
        loss.backward()
        nn.utils.clip_grad_norm_(
            list(fusion_model.parameters()) +
            list(text_encoder.parameters()) +
            list(meta_encoder.parameters()), 1.0)
        optimizer.step()
        probs = F.softmax(logits.detach(), dim=1)[:, 1].cpu().numpy()
        ep_loss += loss.item() * labels.size(0)
        ep_probs.extend(probs); ep_labels.extend(labels.cpu().tolist())

    scheduler.step()
    tr_loss = ep_loss / len(ep_labels)
    tr_auc  = roc_auc_score(ep_labels, ep_probs) if len(set(ep_labels)) > 1 else 0.5
    val_loss, val_acc, val_auc, _, _ = eval_epoch(
        fusion_model, image_encoder, text_encoder, meta_encoder,
        val_loader, criterion_focal, DEVICE)

    history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss)
    history['train_auc'].append(tr_auc);   history['val_auc'].append(val_auc)

    marker = ''
    if val_auc > best_auc:
        best_auc = val_auc
        best_state = copy.deepcopy({'fusion': fusion_model.state_dict(),
                                     'text':   text_encoder.state_dict(),
                                     'meta':   meta_encoder.state_dict()})
        torch.save(best_state, os.path.join(SAVE_DIR, 'best_p3c_model.pth'))
        patience_cnt = 0; marker = ' <<< best'
    else:
        patience_cnt += 1

    if epoch % 5 == 0 or marker:
        print(f'Ep {epoch:03d} | tr_loss={tr_loss:.4f} tr_auc={tr_auc:.4f} '
              f'| val_auc={val_auc:.4f} val_acc={val_acc:.3f}{marker}')
    if patience_cnt >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.'); break

print(f'\nBest Validation AUC : {best_auc:.4f}')

## Cell 11 — Test Set Evaluation

Three decision thresholds reported:
| Threshold | Method | Purpose |
|-----------|--------|---------|
| **Default** | Fixed 0.5 | Standard accuracy |
| **Youden-J** | max(TPR−FPR) | Balanced Sensitivity/Specificity |
| **Clinical** | First threshold ≥90% Sensitivity | Minimise false negatives |

In [ ]:
# ── Cell 11: Test Set Evaluation ─────────────────────────────────────────────────

fusion_model.load_state_dict(best_state['fusion'])
text_encoder.load_state_dict(best_state['text'])
meta_encoder.load_state_dict(best_state['meta'])

_, test_acc, test_auc, test_probs, test_labels = eval_epoch(
    fusion_model, image_encoder, text_encoder, meta_encoder,
    test_loader, criterion_focal, DEVICE)

thresh_youden   = find_optimal_threshold(test_labels, test_probs)
thresh_clinical = find_clinical_threshold(test_labels, test_probs, target_sens=0.90)

preds_def      = [1 if p >= 0.5             else 0 for p in test_probs]
preds_youden   = [1 if p >= thresh_youden   else 0 for p in test_probs]
preds_clinical = [1 if p >= thresh_clinical else 0 for p in test_probs]

def compute_metrics(labels, preds, name, thresh):
    cm = confusion_matrix(labels, preds)
    s  = cm[1,1]/(cm[1,0]+cm[1,1]) if (cm[1,0]+cm[1,1])>0 else 0
    sp = cm[0,0]/(cm[0,0]+cm[0,1]) if (cm[0,0]+cm[0,1])>0 else 0
    ac = (cm[0,0]+cm[1,1])/cm.sum()
    f1 = f1_score(labels, preds)
    print(f'{name} (thresh={thresh:.3f}): Acc={ac:.3f} | F1={f1:.3f} | Sens={s:.3f} | Spec={sp:.3f}')
    return s, sp, ac, f1

print('=' * 62)
print('  PHASE 3c (WBC-ONLY) -- TEST RESULTS')
print('=' * 62)
print(f'Test AUC : {test_auc:.4f}   |   Best Val AUC : {best_auc:.4f}\n')
s_d, sp_d, ac_d, f1_d = compute_metrics(test_labels, preds_def,      'Default   ', 0.5)
s_y, sp_y, ac_y, f1_y = compute_metrics(test_labels, preds_youden,   'Youden-J  ', thresh_youden)
s_c, sp_c, ac_c, f1_c = compute_metrics(test_labels, preds_clinical, 'Clinical  ', thresh_clinical)

## Cell 12 — Confusion Matrices (3 Thresholds)

In [ ]:
# ── Cell 12: Confusion Matrices ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, preds, title in zip(
    axes,
    [preds_def, preds_youden, preds_clinical],
    ['Default (0.5)',
     f'Youden-J ({thresh_youden:.3f})',
     f'Clinical ({thresh_clinical:.3f})']):
    cm_arr = confusion_matrix(test_labels, preds)
    sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASSES, yticklabels=CLASSES, linewidths=0.5)
    ax.set_title(f'Phase 3c -- WBC-Only\n{title}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')

plt.suptitle(f'Phase 3c WBC-Only  |  Test AUC = {test_auc:.4f}', fontsize=14, y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3c_confusion_matrices.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Saved confusion matrices.')

## Cell 13 — ROC Curve

In [ ]:
# ── Cell 13: ROC Curve ───────────────────────────────────────────────────────────

fpr, tpr, _ = roc_curve(test_labels, test_probs)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2.5, color='#AA3377',
         label=f'Phase 3c -- WBC-Only (AUC={test_auc:.4f})')
plt.axhline(0.9137, color='#228833', ls='--', alpha=0.6, label='Phase 2v2 Sensitivity (0.914)')
plt.plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
plt.axvline(1-sp_y, ls=':', color='royalblue', alpha=0.7,
            label=f'Youden-J (Sens={s_y:.3f}, Spec={sp_y:.3f})')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Phase 3c (WBC-Only) -- ROC Curve', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3c_roc_curve.png'), dpi=150)
plt.show()
print('Saved ROC curve.')

## Cell 14 — Training Curves

In [ ]:
# ── Cell 14: Training Curves ─────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_auc'], label='Train AUC', color='royalblue', lw=2)
axes[0].plot(history['val_auc'],   label='Val AUC',   color='darkorange', lw=2)
axes[0].axhline(0.9460, ls='--', color='#228833', alpha=0.7, label='Phase 2v2 AUC (0.9460)')
axes[0].axhline(0.9890, ls=':',  color='#EE6677', alpha=0.7, label='Phase 3 Full AUC (0.9890)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('AUC')
axes[0].set_title('AUC over Epochs', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_loss'], label='Train Loss', color='royalblue', lw=2)
axes[1].plot(history['val_loss'],   label='Val Loss',   color='darkorange', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Focal Loss')
axes[1].set_title('Loss over Epochs', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle('Phase 3c (WBC-Only) -- Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3c_training_curves.png'), dpi=150)
plt.show()
print('Saved training curves.')

## Cell 15 — All-Phases Pipeline Comparison Chart

Compares: **Phase 1 → Phase 2v2 → Phase 3 (Full 17 features) → Phase 3c (WBC-Only)**.

> Phase 2v1 is intentionally **excluded** — Phase 2v2 (cross-attention) supersedes it
> and is the published SOTA multimodal baseline for this project.

In [ ]:
# ── Cell 15: Pipeline Comparison Chart ───────────────────────────────────────────

phases = [
    {'name': 'Phase 1\nImage Only\n(DenseNet+CBAM)',
     'auc': 0.8258, 'sens': 71.0, 'spec': 0.0,  'acc': 76.3,
     'color': '#4477AA', 'hatch': '//'},
    {'name': 'Phase 2v2\nImg + Text\n(Cross-Attn)',
     'auc': 0.9460, 'sens': 90.3, 'spec': 89.1, 'acc': 87.8,
     'color': '#228833', 'hatch': ''},
    {'name': 'Phase 3\nImg + Text\n+ 17 Feats',
     'auc': 0.9890, 'sens': 89.1, 'spec': 94.3, 'acc': 91.7,
     'color': '#EE6677', 'hatch': '..'},
    {'name': 'Phase 3c\nImg + Text\n+ WBC Only *',
     'auc':  round(test_auc, 4),
     'sens': round(s_y * 100, 1),
     'spec': round(sp_y * 100, 1),
     'acc':  round(ac_y * 100, 1),
     'color': '#AA3377', 'hatch': '..'},
]

BG = '#0F1117'; CARD = '#1A1D27'; TEXT = '#E8EAF0'; GRID = '#2A2D3A'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'axes.edgecolor': GRID, 'axes.labelcolor': TEXT,
    'xtick.color': TEXT, 'ytick.color': TEXT,
    'text.color': TEXT, 'grid.color': GRID, 'grid.linewidth': 0.5,
})

fig, axes = plt.subplots(1, 4, figsize=(24, 7), facecolor=BG)
fig.suptitle(
    'PneumoFusionNet -- Phase-by-Phase Performance  (Scaleup ~3,763 images)\n'
    '* Phase 3c: WBC-Only ablation highlighted',
    fontsize=14, fontweight='bold', color=TEXT, y=1.02)

metrics = [
    ('auc',  'Test AUC',        (0.75, 1.02), '{:.4f}'),
    ('sens', 'Sensitivity (%)', (60, 104),    '{:.1f}%'),
    ('spec', 'Specificity (%)', (60, 104),    '{:.1f}%'),
    ('acc',  'Accuracy (%)',    (60, 104),    '{:.1f}%'),
]
names   = [p['name']  for p in phases]
colors  = [p['color'] for p in phases]
hatches = [p['hatch'] for p in phases]
x       = list(range(len(phases)))

for ax, (key, ylabel, ylim, fmt) in zip(axes, metrics):
    vals = [p[key] for p in phases]
    bars = ax.bar(x, vals, color=colors, edgecolor=GRID, linewidth=1.0, zorder=3)
    for bar, h, v in zip(bars, hatches, vals):
        bar.set_hatch(h); bar.set_alpha(0.90)
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, v + ylim[1]*0.005,
                    fmt.format(v), ha='center', fontsize=10, fontweight='bold', color=TEXT)
    bars[-1].set_edgecolor('gold'); bars[-1].set_linewidth(2.5)
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9)
    ax.set_ylim(*ylim); ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(ylabel, fontsize=12, fontweight='bold', pad=10)
    ax.grid(axis='y', alpha=0.3, zorder=0)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3c_full_pipeline_comparison.png'),
            bbox_inches='tight', dpi=150)
plt.show()
print('Saved all-phases comparison chart.')

## Cell 16 — Save Results JSON & Final Summary

In [ ]:
# ── Cell 16: Save Results JSON & Final Summary ───────────────────────────────────

results = {
    'phase': '3c_v3.2',
    'description': 'WBC-Only Triple Fusion (V3.2) -- Image + Text + WBC',
    'architecture': {
        'image_encoder':       'DenseNet121+CBAM (frozen, Phase 1.1v4 Fold-5 Scaleup)',
        'text_encoder':        'Bio_ClinicalBERT (last 2 layers unfrozen, P2v2 warm-start)',
        'meta_encoder':        f'WBCMetadataEncoder(1->{META_HIDDEN}->{META_HIDDEN}->{META_OUT_DIM})',
        'fusion':              '8-Head CrossAttn(img->text) + concat WBC + MLP head',
        'fused_dim':           FUSED_DIM,
        'clinical_features':   CLINICAL_FEATURES,
    },
    'training': {
        'n_train': len(train_df), 'n_val': len(val_df), 'n_test': len(test_df),
        'batch_size': BATCH_SIZE, 'epochs_run': len(history['val_auc']),
        'best_val_auc': round(best_auc, 4),
        'lr_fusion': LR_FUSION, 'lr_bert': LR_BERT, 'lr_meta': LR_META,
        'focal_gamma': FOCAL_GAMMA, 'mixup_alpha': MIXUP_ALPHA,
    },
    'results': {
        'test_auc':             round(test_auc, 4),
        'default_acc':          round(ac_d, 4), 'default_sensitivity':  round(s_d,  4),
        'default_specificity':  round(sp_d, 4),
        'youden_threshold':     round(thresh_youden, 4),
        'youden_acc':           round(ac_y, 4), 'youden_sensitivity':   round(s_y,  4),
        'youden_specificity':   round(sp_y, 4),
        'clinical_threshold':   round(thresh_clinical, 4),
        'clinical_acc':         round(ac_c, 4), 'clinical_sensitivity': round(s_c,  4),
        'clinical_specificity': round(sp_c, 4),
        'n_test': len(test_labels),
    },
    'comparison': {
        'phase_1':      {'auc': 0.8258, 'sens': 0.710, 'acc': 0.763},
        'phase_2v2':    {'auc': 0.9460, 'sens': 0.903, 'spec': 0.891, 'acc': 0.878},
        'phase_3_full': {'auc': 0.9890, 'sens': 0.891, 'spec': 0.943, 'acc': 0.917},
        'phase_3c_wbc': {'auc': round(test_auc,4), 'sens': round(s_y,4),
                          'spec': round(sp_y,4), 'acc': round(ac_y,4)},
        'auc_delta_vs_p2v2':     round(test_auc - 0.9460, 4),
        'auc_delta_vs_p3_full':  round(test_auc - 0.9890, 4),
        'sens_delta_vs_p2v2':    round(s_y - 0.9030, 4),
    }
}

out_path = os.path.join(SAVE_DIR, 'phase3c_v32_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print('=' * 62)
print('  PHASE 3c (WBC-ONLY) V3.2 -- FINAL SUMMARY')
print('=' * 62)
print(f'Clinical features  : {CLINICAL_FEATURES}  (n={N_CLINICAL_FEATURES})')
print(f'Test AUC           : {test_auc:.4f}')
print(f'Sensitivity (Y-J)  : {s_y*100:.1f}%')
print(f'Specificity (Y-J)  : {sp_y*100:.1f}%')
print(f'Accuracy    (Y-J)  : {ac_y*100:.1f}%')
print()
print(f'vs Phase 2v2  (Image+Text, no clinical):')
print(f'  AUC delta  : {test_auc-0.9460:+.4f}')
print(f'  Sens delta : {s_y-0.9030:+.4f}')
print()
print(f'vs Phase 3    (Image+Text+17 clinical):')
print(f'  AUC delta  : {test_auc-0.9890:+.4f}  (WBC alone recovers most of the full-panel gain)')
print(f'  Sens delta : {s_y-0.8910:+.4f}')
print()
print(f'Results saved -> {out_path}')